In [1]:
import requests
import json
from datetime import datetime
from pathlib import Path

# ABN AMRO

In [28]:
SOURCE_URL = "https://www.werkenbijabnamro.nl/en/api/vacancy/"
SOURCE_URL = "https://www.werkenbijbdo.nl/api/vacancy/"
SOURCE_URL = "https://www.werkenbijrandstad.nl/api/vacancy/"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Firefox/152.0",
    "Accept": "application/json, text/plain, */*",
    "X-Requested-With": "XMLHttpRequest",
    "Accept-Language": "en-US,en;q=0.9",
}

In [29]:
params = {"sort": "created", "sortDir": "DESC", "pageNumber": 1}
r = requests.get(SOURCE_URL, params=params, headers=HEADERS, timeout=20)
print("status:", r.status_code)

data = r.json()
print("top-level keys:", list(data.keys()))
print("meta:", data["meta"]["num_total_hits"], "jobs across",
      data["meta"]["totalPageCount"], "pages")
print("first job:", data["vacancies"][0]["title"])

status: 200
top-level keys: ['vacancies', 'facets', 'meta']
meta: 50 jobs across 5 pages
first job: intercedent Belastingdienst


In [30]:
def first_id(page):
    p = {"sort": "created", "sortDir": "DESC", "pageNumber": page}
    d = requests.get(SOURCE_URL, params=p, headers=HEADERS, timeout=20).json()
    return d["vacancies"][0]["id"]

print("page 1 first id:", first_id(1))
print("page 2 first id:", first_id(2))

page 1 first id: 1638
page 2 first id: 1629


In [31]:
all_jobs = []
total_pages = data["meta"]["totalPageCount"]

for page in range(1, total_pages + 1):
    p = {"sort": "created", "sortDir": "DESC", "pageNumber": page}
    d = requests.get(SOURCE_URL, params=p, headers=HEADERS, timeout=20).json()
    all_jobs.extend(d["vacancies"])
    print(f"page {page}: +{len(d['vacancies'])} (running total {len(all_jobs)})")

# dedupe by id, then check against what meta promised
unique = {j["id"]: j for j in all_jobs}
print("\ncollected unique:", len(unique), "| meta said:", data["meta"]["num_total_hits"])

page 1: +10 (running total 10)
page 2: +10 (running total 20)
page 3: +10 (running total 30)
page 4: +10 (running total 40)
page 5: +10 (running total 50)

collected unique: 50 | meta said: 50


In [32]:
for j in list(unique.values())[:8]:
    cats = ", ".join(o["value"] for o in j.get("option_values", []))
    print(f"- {j['title']}  |  {j['city']}  |  {cats} | {str(j['created']).split('T')[0]}")

- intercedent Belastingdienst  |  Eindhoven  |  32 - 40, HBO, €2750 - €3200 | 2026-06-29
- operational manager  |  Eindhoven  |  32 - 40, HBO, €3.398 - €6.492 | 2026-06-26
- hr medewerker  |  Oostrum (L)  |  32 - 40, € 2.599 - € 3.757 | 2026-06-26
- Intercedent Onderwijs  |  Meppel  |  40, HBO, €2.750 - €3.200 | 2026-06-26
- senior administrative and accountsupport employee  |  Utrecht  |  32 - 40, MBO, €2.706 - €3.757 | 2026-06-26
- senior administrative and accountsupport employee  |  Amsterdam  |  32 - 40, MBO, €2.706 - €3.757 | 2026-06-26
- medewerker administratie en support  |  Eindhoven  |  32 - 40, MBO, €2.706 - €3.757 | 2026-06-26
- junior accountmanager  |  Rotterdam  |  32 - 40, HBO, € 2.750 - € 3.200 | 2026-06-26
